In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from scripts.utils.load_model import load_model, ModelName
from scripts.utils.load_dataset import load_dataset


model, dataset_transform = load_model(ModelName.SALMAN_RESNET50, dataset_name="imagenet", device="cuda:0")
dl_train, dl_eval, dl_test = load_dataset("imagenet", transform=dataset_transform, train_batch=128, eval_batch=128)

In [ ]:
from ulib import StopCriteria, PertModule
from ulib.attacks.du_attack import DU_ATTACK

stop = StopCriteria(max_epochs=10, max_time=600)
pert_model = PertModule(model, data_shape=dl_train.get_tensor(0).shape[1:], eps=8 / 255)

attack = DU_ATTACK(
    pert_model=pert_model,
    eval_freq=10,
    alpha=0.2,
    beta=0.9,
    steps=30,
)

pert = attack.fit(dl_train, dl_eval, stop)

In [ ]:
from ulib.evaluator import ExtendedEvaluator
from ulib.utils.visualize import display_pert

evaluator = ExtendedEvaluator(pert_model, verbose=True)
print(evaluator.evaluate(dl_eval))
display_pert(pert_model)